# Full-Pol PolSAR Decomposition Tutorial

This notebook demonstrates five polarimetric decompositions on **quad-polarization** SAR data
(HH, HV, VH, VV) using GRDL.  Each decomposition provides a physically interpretable
breakdown of the measured scattering.

**Algorithms demonstrated:**
1. **Pauli decomposition** — coherent, pixel-by-pixel basis transform
2. **FullPolHAalpha** — Cloude-Pottier eigendecomposition (H/A/α)
3. **FreemanDurden3C** — model-based 3-component power decomposition
4. **ModelFree3C (MF3CF)** — model-free 3-component decomposition (Dey et al. 2020)
5. **ModelFree4C (MF4CF)** — model-free 4-component with helix power (Dey et al. 2021)
6. Vegetation indices (RVI, GRVI, PRVI) — placeholder

**Data:** NISAR L1 RSLC quad-pol chip or synthetic fallback.

> **Dual-pol H/Alpha:** see `dual_pol_decomposition_tutorial.ipynb`.

---

## Theory Overview

### Pauli Decomposition

The 2×2 scattering matrix $S$ is expressed in the Pauli basis:

$$S = a P_1 + b P_2 + c P_3$$

$$a = \frac{S_{HH}+S_{VV}}{\sqrt{2}}, \quad b = \frac{S_{HH}-S_{VV}}{\sqrt{2}}, \quad c = \frac{S_{HV}+S_{VH}}{\sqrt{2}}$$

RGB mapping: **R** = $|b|$ (double-bounce), **G** = $|c|$ (volume), **B** = $|a|$ (surface).

### Cloude-Pottier H/A/α (FullPolHAalpha)

Eigendecomposition of the 3×3 coherency matrix $[T_3]$:

$$[T_3] = \sum_{i=1}^{3} \lambda_i u_i u_i^H$$

Yields entropy $H$, anisotropy $A$, and mean scattering angle $\alpha$.
See `dual_pol_decomposition_tutorial.ipynb` for full equations.

### Freeman-Durden 3C

Model-based decomposition of $[C_3]$ into surface ($f_s$), double-bounce ($f_d$),
and volume ($f_v$) scattering contributions:

$$[C_3] = f_s [C_3]^{\text{surface}} + f_d [C_3]^{\text{dbl}} + f_v [C_3]^{\text{vol}}$$

RGB mapping: **R** = $P_d$ (double-bounce), **G** = $P_v$ (volume), **B** = $P_s$ (surface).

**Reference:** Freeman, A. and Durden, S.L. (1998). IEEE Trans. Geoscience and Remote Sensing, 36(3), 963–973.

### Model-Free Decompositions (MF3CF / MF4CF)

Dey et al. use the degree of polarisation $m$ and a rotation angle $\theta_{FP}$ derived
directly from $[T_3]$ elements, requiring no scene-model assumptions:

$$\theta_{FP} = \frac{1}{2}\arctan\frac{T_{22}+T_{33}-T_{11}}{T_{11}-T_{22}+T_{33}}, \quad
m = \sqrt{1 - \frac{27 \det[T_3]}{\mathrm{tr}^3[T_3]}}$$

MF4CF adds helix power via a second angle $\tau_{FP}$.

**References:**
- Dey, S. et al. (2020). *Model-free four-component scattering power decomposition.* IEEE GRSL, 17(11).
- Dey, S. et al. (2021). *Geometry of the model-free scattering power.* IEEE GRSL.

In [1]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
import gc

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")


grdl 0.6.2 — ready


In [2]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from grdl.IO.sar.nisar import NISARReader
from grdl.image_processing.decomposition import (
    PauliDecomposition, FullPolHAalpha,
    FreemanDurden3C, ModelFree3C, ModelFree4C,
)
from grdk.viewers.dual_viewer import DualGeoViewer

%gui qt6
print('Imports OK')


Imports OK


In [3]:
# =============================================================================
# Configuration
# =============================================================================
nisar_file = Path(
    '/data/sar/slc/nisar/l1_rslc/'
    '20260105T055924_20260105T055931/'
    'NISAR_L1_PR_RSLC_009_116_D_054_2005_QPDH_A_20260105T055924_20260105T055931'
    '_X05010_N_P_J_001.h5'
)
frequency    = 'A'
polarizations = 'all'
chip_size    = 1024

window_size  = 7     # T3/C3 spatial averaging window

In [ ]:
# =============================================================================
# Load NISAR quad-pol or generate synthetic fallback
# =============================================================================
def _synthetic_quad_pol(rows=512, cols=512, seed=42):
    """Simple quad-pol SLC with scene regions."""
    rng = np.random.default_rng(seed)
    shh = (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols))).astype(np.complex64)
    shv = (0.3 * (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols)))).astype(np.complex64)
    svh = shv.copy()
    svv = (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols))).astype(np.complex64)
    # Urban block: enhance double-bounce (HH+VV out-of-phase)
    r0, r1, c0, c1 = rows//8, rows*3//8, cols//8, cols*3//8
    shh[r0:r1, c0:c1] *= 2.0
    svv[r0:r1, c0:c1] *= -2.0
    shv[r0:r1, c0:c1] *= 0.2
    # Vegetation: high cross-pol
    r0, r1, c0, c1 = rows*5//8, rows*7//8, cols*5//8, cols*7//8
    shv[r0:r1, c0:c1] *= 4.0
    svh[r0:r1, c0:c1] *= 4.0
    return shh, shv, svh, svv

if nisar_file.exists():
    reader = NISARReader(nisar_file, frequency=frequency, polarizations=polarizations)
    meta = reader.metadata
    N = chip_size
    r0 = max(0, (meta.rows - N) // 2)
    c0 = max(0, (meta.cols - N) // 2)
    cube = reader.read_chip(r0, r0 + N, c0, c0 + N)
    reader.close()
    pol_index = {cm.polarization: i for i, cm in enumerate(meta.channel_metadata)}
    shh = cube[pol_index['HH']]
    shv = cube[pol_index['HV']]
    svh = cube[pol_index.get('VH', pol_index['HV'])]
    svv = cube[pol_index['VV']]
    print(f'NISAR chip: {shh.shape}')
    del cube
    gc.collect()

else:
    N = chip_size
    shh, shv, svh, svv = _synthetic_quad_pol(rows=N, cols=N)
    print(f'Synthetic quad-pol: {shh.shape}')

SyntaxError: invalid syntax (1565797909.py, line 39)

---
## 1. Pauli Decomposition — Coherent Scattering Components

The Pauli decomposition is a pixel-by-pixel (single-look) transform.  No spatial averaging
is applied; the result shows instantaneous scattering mechanism at full resolution with
full speckle.

In [ ]:
# =============================================================================
# Pauli decomposition
# =============================================================================
pauli = PauliDecomposition()
pauli_c = pauli.decompose(shh, shv, svh, svv)
# Keys: 'surface', 'double_bounce', 'volume' — complex Pauli coefficients
print(f'Pauli components: {list(pauli_c.keys())}')

def stretch(x, pct=(2, 98)):
    lo, hi = np.percentile(x, pct)
    return np.clip((x - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)

# Standard PolSAR Pauli RGB: R=double-bounce, G=volume, B=surface
rgb_pauli = np.stack([
    stretch(np.abs(pauli_c['double_bounce'])),
    stretch(np.abs(pauli_c['volume'])),
    stretch(np.abs(pauli_c['surface'])),
], axis=0)
print(f'Pauli RGB: {rgb_pauli.shape}  (R=double-bounce, G=volume, B=surface)')

In [ ]:
# =============================================================================
# GRDK viewer — Pauli decomposition
# =============================================================================
# Left: raw intensity (R=HH, G=HV, B=VV), Right: Pauli RGB
rgb_raw = np.stack([
    stretch(np.abs(shh)),
    stretch(np.abs(shv)),
    stretch(np.abs(svv)),
], axis=0)

viewer_pauli = DualGeoViewer()
viewer_pauli.set_mode('dual')
viewer_pauli.setWindowTitle('Pauli — Raw (R=HH,G=HV,B=VV) (L) vs Pauli RGB (R)')
viewer_pauli.set_array(rgb_raw, pane=0)
viewer_pauli.set_array(rgb_pauli, pane=1)
viewer_pauli.show()
print('Viewer: Pauli')

---
## 2. FullPolHAalpha — Cloude-Pottier Eigendecomposition

Eigendecomposition of the spatially-averaged $[T_3]$ coherency matrix.  The window
size controls the speckle-vs-resolution trade-off.

In [ ]:
# =============================================================================
# FullPolHAalpha
# =============================================================================
fp_ha = FullPolHAalpha(window_size=window_size)
ha_c = fp_ha.decompose(shh, shv, svh, svv)
print(f'FullPolHAalpha keys: {list(ha_c.keys())}')
print(f'H   range: [{ha_c["entropy"].min():.3f}, {ha_c["entropy"].max():.3f}]')
print(f'α   range: [{ha_c["alpha"].min():.1f}°, {ha_c["alpha"].max():.1f}°]')
print(f'A   range: [{ha_c["anisotropy"].min():.3f}, {ha_c["anisotropy"].max():.3f}]')

rgb_ha, ha_meta = fp_ha.to_rgb(ha_c)
print(f'RGB: {rgb_ha.shape}  ({" / ".join(cm.name for cm in ha_meta.channel_metadata)})')

In [ ]:
# =============================================================================
# GRDK viewer — H/A/Alpha
# =============================================================================
viewer_ha = DualGeoViewer()
viewer_ha.set_mode('dual')
viewer_ha.setWindowTitle(f'FullPolHAalpha (win={window_size}) — Pauli (L) vs H/A/α RGB (R)')
viewer_ha.set_array(rgb_pauli, pane=0)
viewer_ha.set_array(rgb_ha, pane=1)
viewer_ha.show()
print('Viewer: FullPolHAalpha')

In [ ]:
# =============================================================================
# H-Alpha plane (Cloude-Pottier classification diagram)
# =============================================================================
# Reference: Cloude & Pottier (1997), IEEE TGRS 35(1), Fig. 4
#
# BASIS: [T3] Pauli coherency matrix — k=(HH+VV, HH−VV, 2HV)/√2.
#   alpha = arccos(|e_i[0]|) where e_i[0] is the (HH+VV)/√2 component.
#   alpha≈0°  → surface / odd-bounce (HH+VV dominates)
#   alpha≈45° → volume / random (HV cross-pol dominates)
#   alpha≈90° → double-bounce (HH−VV dominates, dihedral reflector)
#
# For L-band NISAR data over vegetation or urban terrain, alpha=40–85° is
# physically expected: L-band penetrates canopy creating volume scattering
# (alpha≈45–60°) and double-bounce from tree trunks and structures
# (alpha≈60–90°).  This is NOT comparable to the dual-pol alpha, which
# is computed in the [C2] lexicographic (HH, HV) basis.
H_flat     = ha_c['entropy'].ravel()
alpha_flat = ha_c['alpha'].ravel()

mask_ha = np.isfinite(H_flat) & np.isfinite(alpha_flat)
H_valid     = H_flat[mask_ha]
alpha_valid = alpha_flat[mask_ha]

fig, ax = plt.subplots(figsize=(8, 6))

hb = ax.hexbin(H_valid, alpha_valid, gridsize=120, cmap='inferno',
               mincnt=1, extent=[0, 1, 0, 90])
fig.colorbar(hb, ax=ax, label='Pixel count')

# Zone boundaries (Cloude-Pottier 9-zone scheme)
for h_line in [0.5, 0.9]:
    ax.axvline(h_line, color='gray', linewidth=0.8, linestyle='--', alpha=0.7)
for a_line in [42.5, 47.5]:
    ax.axhline(a_line, color='gray', linewidth=0.8, linestyle='--', alpha=0.7)

zone_labels = {
    (0.25, 21):  'Z9\nSurface',
    (0.25, 45):  'Z8\nDipole',
    (0.25, 69):  'Z7\nDihedral',
    (0.70, 21):  'Z6',
    (0.70, 45):  'Z5\nVegetation',
    (0.70, 69):  'Z4',
    (0.95, 21):  'Z3',
    (0.95, 45):  'Z2',
    (0.95, 69):  'Z1',
}
for (hpos, apos), label in zone_labels.items():
    ax.text(hpos, apos, label, color='cyan', fontsize=7,
            ha='center', va='center', alpha=0.8)

ax.set_xlabel('Entropy (H)')
ax.set_ylabel('Alpha (°)  [T3 Pauli basis]')
ax.set_title('H-α Plane — Cloude-Pottier (full-pol, [T3] Pauli basis)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 90)
plt.tight_layout()
plt.show()

print(f'Valid pixels: {mask_ha.sum():,} / {mask_ha.size:,}')


---
## 3. Freeman-Durden 3C — Model-Based Power Decomposition

Decomposes $[C_3]$ into surface, double-bounce, and volume contributions assuming
specific electromagnetic scattering models for each component.  Works best for
scenes with clear mechanism separation (low or medium rotation).

In [ ]:
# =============================================================================
# FreemanDurden3C
# =============================================================================
fd = FreemanDurden3C(window_size=window_size)
fd_c = fd.decompose(shh, shv, svh, svv)
print(f'FreemanDurden3C keys: {list(fd_c.keys())}')
print(f'Ps range: [{fd_c["surface"].min():.4f}, {fd_c["surface"].max():.4f}]')
print(f'Pd range: [{fd_c["double_bounce"].min():.4f}, {fd_c["double_bounce"].max():.4f}]')
print(f'Pv range: [{fd_c["volume"].min():.4f}, {fd_c["volume"].max():.4f}]')

rgb_fd, fd_meta = fd.to_rgb(fd_c)
print(f'RGB: {rgb_fd.shape}  ({" / ".join(cm.name for cm in fd_meta.channel_metadata)})')

In [ ]:
# =============================================================================
# GRDK viewer — Freeman-Durden
# =============================================================================
viewer_fd = DualGeoViewer()
viewer_fd.set_mode('dual')
viewer_fd.setWindowTitle(f'FreemanDurden3C (win={window_size}) — Pauli (L) vs FD RGB (R)')
viewer_fd.set_array(rgb_pauli, pane=0)
viewer_fd.set_array(rgb_fd, pane=1)
viewer_fd.show()
print('Viewer: FreemanDurden3C')

---
## 4. Model-Free Decompositions — MF3CF and MF4CF

Unlike Freeman-Durden, the model-free decompositions use the degree of polarisation $m$
and the target orientation angle $\theta_{FP}$ computed directly from $[T_3]$, with no
a priori scene model.  MF4CF adds a helix power component $P_c$ via the ellipticity
angle $\tau_{FP}$.

In [ ]:
# =============================================================================
# ModelFree3C (MF3CF)
# =============================================================================
mf3 = ModelFree3C(window_size=window_size)
mf3_c = mf3.decompose(shh, shv, svh, svv)
print(f'MF3CF keys: {list(mf3_c.keys())}')

rgb_mf3, mf3_meta = mf3.to_rgb(mf3_c)

# =============================================================================
# ModelFree4C (MF4CF)
# =============================================================================
mf4 = ModelFree4C(window_size=window_size)
mf4_c = mf4.decompose(shh, shv, svh, svv)
print(f'MF4CF keys: {list(mf4_c.keys())}')
print(f'Helix Pc range: [{mf4_c["helix"].min():.4f}, {mf4_c["helix"].max():.4f}]')

rgb_mf4, mf4_meta = mf4.to_rgb(mf4_c)
print(f'MF3CF RGB: {rgb_mf3.shape},  MF4CF RGB: {rgb_mf4.shape}')

In [ ]:
# =============================================================================
# GRDK viewer — MF3CF vs MF4CF
# =============================================================================
viewer_mf = DualGeoViewer()
viewer_mf.set_mode('dual')
viewer_mf.setWindowTitle(f'Model-Free — MF3CF (L) vs MF4CF (R)  (win={window_size})')
viewer_mf.set_array(rgb_mf3, pane=0)
viewer_mf.set_array(rgb_mf4, pane=1)
viewer_mf.show()
print('Viewer: MF3CF vs MF4CF')

---
## 5. Vegetation Indices — RVI, GRVI, PRVI (Placeholder)

Polarimetric vegetation indices use the polarimetric entropy, span, and eigenvalues to
estimate vegetation cover, canopy structure, and biomass.

| Index | Formula | Reference |
|-------|---------|----------|
| RVI   | $4\sigma_{HV}^0 / (\sigma_{HH}^0 + \sigma_{VV}^0 + 2\sigma_{HV}^0)$ | Kim & van Zyl 2009 |
| GRVI  | $1 - GD_{VI}$ | Chandrasekar et al. 2010 |
| PRVI  | $(1 - \lambda_1/(\lambda_1+\lambda_2+\lambda_3)) \cdot \sigma_{VV}^0$ | Raney et al. 2011 |

**Not yet implemented** (Phase 5 of the PolSAR expansion plan).

In [ ]:
# =============================================================================
# Vegetation Indices — placeholder
# =============================================================================

# from grdl.image_processing.decomposition import RVI, GRVI, PRVI
#
# rvi  = RVI(window_size=window_size).decompose(shh, shv, svh, svv)
# grvi = GRVI(window_size=window_size).decompose(shh, shv, svh, svv)
# prvi = PRVI(window_size=window_size).decompose(shh, shv, svh, svv)
#
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for ax, (name, data) in zip(axes, [('RVI', rvi['rvi']), ('GRVI', grvi['grvi']), ('PRVI', prvi['prvi'])]):
#     ax.imshow(data, vmin=0, vmax=1, cmap='YlGn')
#     ax.set_title(name)
#     ax.axis('off')
# plt.tight_layout()
# plt.show()

print('Vegetation indices (RVI, GRVI, PRVI) not yet implemented — placeholder only')